# Quantitative Plots
> Heatmaps and dotplots for isoform quantification visualization.

This module provides standalone and integrated heatmap/dotplot functions for visualizing isoform expression patterns across groups.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| default_exp quant_plots

In [ ]:
#| export
from __future__ import annotations

from typing import List, Optional, Tuple, Dict
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import textwrap

try:
    from scipy.sparse import issparse
except Exception:
    def issparse(_):  # type: ignore
        return False


def _dense_X(X) -> np.ndarray:
    """Return a dense float array from (dense | sparse) matrix-like."""
    return np.asarray(X.toarray() if issparse(X) else X, dtype=float)


def _wrap_labels(labels: List[str], width: int = 14) -> List[str]:
    """Wrap labels for x-axis ticks."""
    def _wrap(s: str) -> str:
        return "\n".join(
            textwrap.wrap(
                str(s),
                width=max(4, int(width)),
                break_long_words=False,
            )
        )
    return [_wrap(str(x)) for x in labels]


def _resolve_transcripts(
    adata,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    group_col: Optional[str] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
) -> List[str]:
    """
    Resolve which transcripts to plot.

    Either takes explicit transcript list OR derives top_n transcripts
    from gene_id using group-based ranking.

    Parameters
    ----------
    adata : AnnData
        Annotated data object
    transcripts : list of str, optional
        Explicit list of transcript IDs
    gene_id : str, optional
        Gene ID to get top transcripts from
    top_n : int
        Number of top transcripts to select (when using gene_id)
    group_col : str, optional
        Column in adata.obs for grouping (required when using gene_id)
    estimator : str
        Estimator for PSI calculation ('pseudobulk' or 'mean')
    dirichlet_alpha : float
        Dirichlet alpha for pseudobulk estimation
    epsilon : float
        Small value to avoid division by zero

    Returns
    -------
    List[str]
        List of transcript IDs to plot
    """
    if transcripts is not None:
        # Path 1: User provided explicit list
        return transcripts if isinstance(transcripts, list) else [transcripts]
    elif gene_id is not None:
        # Path 2: Get top_n transcripts for this gene
        if group_col is None:
            raise ValueError("group_col is required when using gene_id")

        iso_ids, groups, V = _compute_group_matrix_from_adata(
            adata, gene_id, group_col,
            top_n=top_n, estimator=estimator,
            dirichlet_alpha=dirichlet_alpha, epsilon=epsilon
        )
        return iso_ids
    else:
        raise ValueError("Must provide either 'transcripts' or 'gene_id'")


def _compute_group_matrix_from_adata(
    adata,
    gene_id: str,
    group_col: str,
    *,
    top_n: Optional[int] = 2,
    estimator: str = "pseudobulk",      # 'pseudobulk'|'dirichlet'|'cell-mean'|'cell-median'|'coverage-weighted'
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
) -> Tuple[List[str], List[str], np.ndarray]:
    """
    Compute an isoform×group PSI-like matrix from AnnData where:
      - adata.var['geneId'] maps isoforms/transcripts to gene IDs
      - adata.X are counts (cells × isoforms)

    Returns:
      iso_ids : list[str]        selected isoform ids (rows)
      groups  : list[str]        group names in stable order (cols)
      V       : np.ndarray       shape (n_iso, n_groups)
    """
    if adata is None:
        return [], [], np.zeros((0, 0), float)
    if "geneId" not in adata.var:
        return [], [], np.zeros((0, 0), float)

    mask = adata.var["geneId"].astype(str).values == str(gene_id)
    if not np.any(mask):
        return [], [], np.zeros((0, 0), float)

    iso_ids = adata.var_names[mask]
    X = _dense_X(adata.X)

    groups_all = adata.obs[group_col].astype(str).values
    groups = list(dict.fromkeys(groups_all))  # stable order

    G = X[:, mask]  # cells × isoforms-of-gene
    lib = G.sum(1) + float(epsilon)

    def _psi_cellwise(arr: np.ndarray) -> np.ndarray:
        return np.divide(
            arr,
            lib[:, None],
            out=np.zeros_like(arr, float),
            where=(lib[:, None] > 0),
        )

    V_rows: List[np.ndarray] = []

    if estimator in ("cell-mean", "cell-median", "coverage-weighted"):
        PSI = _psi_cellwise(G)
        weights = lib if estimator == "coverage-weighted" else None
        for g in groups:
            m = (groups_all == g)
            if not m.any():
                V_rows.append(np.zeros(G.shape[1], float))
                continue
            A = PSI[m]
            if estimator == "cell-median":
                V_rows.append(np.nanmedian(A, axis=0))
            elif estimator == "coverage-weighted":
                w = weights[m]
                w = w / (w.sum() + 1e-12)
                V_rows.append((A * w[:, None]).sum(0))
            else:
                V_rows.append(np.nanmean(A, axis=0))

    elif estimator in ("pseudobulk", "dirichlet"):
        for g in groups:
            m = (groups_all == g)
            if not m.any():
                V_rows.append(np.zeros(G.shape[1], float))
                continue
            S = G[m].sum(0)
            if estimator == "dirichlet":
                S = S + float(dirichlet_alpha)
            denom = float(S.sum()) + float(epsilon)
            V_rows.append((S / denom).ravel())
    else:
        raise ValueError(
            f"Unknown estimator='{estimator}'. "
            "Use: pseudobulk|dirichlet|cell-mean|cell-median|coverage-weighted."
        )

    V = np.vstack(V_rows).T  # isoforms × groups

    if top_n is not None and int(top_n) < len(iso_ids) and V.size:
        k = int(top_n)
        keep_idx = np.argsort(V.mean(1))[::-1][:k]
        iso_ids = iso_ids[keep_idx]
        V = V[keep_idx, :]

    return iso_ids.tolist(), groups, V


def _compute_dot_matrices_from_adata(
    adata,
    gene_id: str,
    group_col: str,
    *,
    top_n: Optional[int] = 2,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    size_mode: str = "prevalence",
    psi_threshold: float = 0.05,
) -> Tuple[List[str], List[str], np.ndarray, np.ndarray]:
    """
    Dotplot matrices:
      - dot_color: isoform×group mean PSI-like (same as heatmap)
      - dot_size:  isoform×group size in [0,1] (default: prevalence)

    Returns:
      iso_ids, groups, V (color), S (size)
    """
    iso_ids, groups, V = _compute_group_matrix_from_adata(
        adata,
        gene_id,
        group_col,
        top_n=top_n,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
        epsilon=epsilon,
    )
    if not iso_ids or V.size == 0:
        return iso_ids, groups, V, np.zeros((0, 0), float)

    if size_mode != "prevalence":
        raise ValueError(f"size_mode='{size_mode}' not implemented. Use 'prevalence'.")

    mask_full = adata.var["geneId"].astype(str).values == str(gene_id)
    iso_all = adata.var_names[mask_full]
    pos = {tid: i for i, tid in enumerate(iso_all)}
    idx = np.array([pos[t] for t in iso_ids], dtype=int)

    X = _dense_X(adata.X)
    G_all = X[:, mask_full]
    G = G_all[:, idx]
    lib = G.sum(1) + float(epsilon)

    groups_all = adata.obs[group_col].astype(str).values
    PSI = np.divide(
        G,
        lib[:, None],
        out=np.zeros_like(G, float),
        where=(lib[:, None] > 0),
    )

    S_rows: List[np.ndarray] = []
    thr = float(psi_threshold)
    for g in groups:
        m = (groups_all == g)
        if not m.any():
            S_rows.append(np.zeros(G.shape[1], float))
            continue
        S_rows.append((PSI[m] > thr).mean(0))

    S = np.vstack(S_rows).T
    return iso_ids, groups, V, S


def _standalone_heatmap(
    V: np.ndarray,
    *,
    row_labels: List[str],
    col_labels: List[str],
    title: str = "",
    xlabel: str = "",
    cmap: str = "magma",
    colorbar_label: str = "PSI",
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
    label_wrap: int = 14,
    label_rot: int = 35,
    show_colorbar: bool = False,
) -> plt.Figure:
    n_rows, n_cols = V.shape
    H = fig_height if fig_height is not None else max(3.5, 0.35 * n_rows + 1.8)
    fig, ax = plt.subplots(figsize=(fig_width, H))
    im = ax.imshow(
        V,
        aspect="auto",
        interpolation="nearest",
        origin="lower",
        cmap=mpl.cm.get_cmap(cmap),
    )

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(
        _wrap_labels(col_labels, width=label_wrap),
        rotation=int(label_rot),
        ha="right",
        va="top",
        fontsize=9,
    )
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(row_labels, fontsize=9)

    ax.set_title(title, fontsize=12)
    if xlabel:
        ax.set_xlabel(xlabel)

    if show_colorbar:
        cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
        cb.set_label(colorbar_label)

    fig.tight_layout()
    return fig


def _standalone_dotplot(
    C: np.ndarray,
    S: np.ndarray,
    *,
    row_labels: List[str],
    col_labels: List[str],
    title: str = "",
    xlabel: str = "",
    cmap: str = "magma",
    colorbar_label: str = "PSI",
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
    label_wrap: int = 14,
    label_rot: int = 35,
    show_colorbar: bool = False,
) -> plt.Figure:
    n_rows, n_cols = C.shape
    if S.shape != C.shape:
        raise ValueError("dot_size (S) must have same shape as dot_color (C).")

    H = fig_height if fig_height is not None else max(3.5, 0.35 * n_rows + 1.8)
    fig, ax = plt.subplots(figsize=(fig_width, H))

    s = np.clip(np.asarray(S, float), 0.0, 1.0)
    sizes = 16.0 + s * (180.0 - 16.0)

    Xg, Yg = np.meshgrid(np.arange(n_cols), np.arange(n_rows))
    sc = ax.scatter(
        Xg.ravel(),
        Yg.ravel(),
        c=np.asarray(C, float).ravel(),
        s=sizes.ravel(),
        cmap=mpl.cm.get_cmap(cmap),
        edgecolors="#333333",
        linewidths=0.3,
    )

    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(-0.5, n_rows - 0.5)
    ax.invert_yaxis()

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(
        _wrap_labels(col_labels, width=label_wrap),
        rotation=int(label_rot),
        ha="right",
        va="top",
        fontsize=9,
    )
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(row_labels, fontsize=9)

    ax.set_title(title, fontsize=12)
    if xlabel:
        ax.set_xlabel(xlabel)

    if show_colorbar:
        cb = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
        cb.set_label(colorbar_label)

    fig.tight_layout()
    return fig

# Examples will be added after function definitions

## Standalone Plotting Functions

These functions work directly with computed matrices (no TranscriptData needed).

In [ ]:
#| export
def _compute_replicate_psi_from_adata(
    adata,
    group_col: str,
    replicate_col: str,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    epsilon: float = 1e-6,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
) -> Tuple[List[str], List[str], Dict[Tuple[int, int], np.ndarray]]:
    """
    Compute per-replicate PSI values for a gene's isoforms across groups.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    replicate_col : str
        Column in adata.obs defining replicates (e.g., 'batch', 'sample_id')
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot.
        Mutually exclusive with gene_id + top_n.
    gene_id : str, optional
        Gene ID to analyze. Used with top_n to select isoforms.
    top_n : int
        Number of top isoforms to include (when using gene_id)
    epsilon : float
        Small value to avoid division by zero
    estimator : str
        PSI estimator ('pseudobulk', 'dirichlet', 'cell-mean', 'cell-median', 'coverage-weighted')
    dirichlet_alpha : float
        Dirichlet alpha parameter (for dirichlet estimator)
        
    Returns
    -------
    iso_ids : List[str]
        List of isoform IDs (top_n isoforms)
    groups : List[str]
        List of group names
    replicate_data : Dict[Tuple[int, int], np.ndarray]
        Dictionary mapping (isoform_idx, group_idx) -> array of PSI values per replicate
    """
    # Resolve transcripts using the standard pattern
    iso_ids = _resolve_transcripts(
        adata,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        group_col=group_col,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
        epsilon=epsilon,
    )
    
    if not iso_ids:
        return [], [], {}
    
    # Create mask for these specific transcripts
    iso_mask = np.isin(adata.var_names, iso_ids)
    if not iso_mask.any():
        return [], [], {}
    
    # Map each iso_id to its column position in X[:, iso_mask].
    # iso_mask selects columns in adata.var_names order, which may differ
    # from iso_ids order (iso_ids is sorted by PSI descending).
    var_names_in_mask = adata.var_names[iso_mask].tolist()
    
    X = _dense_X(adata.X)
    obs = adata.obs
    group_vals = obs[group_col].astype(str).values
    groups = list(dict.fromkeys(group_vals))
    
    # Compute per-replicate PSI for each isoform and group
    replicate_data: Dict[Tuple[int, int], np.ndarray] = {}
    for j, g in enumerate(groups):
        m_group = (group_vals == g)
        if not m_group.any():
            for i, _ in enumerate(iso_ids):
                replicate_data[(i, j)] = np.array([], float)
            continue
        
        rep_vals = obs.loc[m_group, replicate_col].astype(str).values
        uniq_reps = list(dict.fromkeys(rep_vals))
        
        for i, tid in enumerate(iso_ids):
            vals = []
            iso_idx_in_subset = var_names_in_mask.index(tid)
            
            for r in uniq_reps:
                m_rep = m_group & (obs[replicate_col].astype(str).values == r)
                if not m_rep.any():
                    continue
                
                # Get all counts for transcripts in this set
                Xg = X[m_rep][:, iso_mask]
                
                if estimator in ("cell-mean", "cell-median", "coverage-weighted"):
                    lib = Xg.sum(1) + epsilon
                    psi = np.divide(Xg, lib[:, None],
                                    out=np.zeros_like(Xg, float),
                                    where=(lib[:, None] > 0))
                    v = psi[:, iso_idx_in_subset]
                    if estimator == "cell-median":
                        vals.append(float(np.nanmedian(v)))
                    elif estimator == "coverage-weighted":
                        w = lib / (lib.sum() + 1e-12)
                        vals.append(float((v * w).sum()))
                    else:
                        vals.append(float(np.nanmean(v)))
                        
                elif estimator in ("pseudobulk", "dirichlet"):
                    S = Xg.sum(0)
                    if estimator == "dirichlet":
                        S = S + dirichlet_alpha
                    denom = float(S.sum()) + epsilon
                    v = float(S[iso_idx_in_subset] / denom) if denom > 0 else 0.0
                    vals.append(v)
                else:
                    raise ValueError(f"Unknown estimator: {estimator}")
                    
            replicate_data[(i, j)] = np.array(vals, float) if len(vals) else np.array([], float)
    
    return iso_ids, groups, replicate_data


def _standalone_replicate_plot(
    replicate_data: Dict[Tuple[int, int], np.ndarray],
    *,
    group_labels: List[str],
    isoform_labels: List[str],
    title: str = "",
    xlabel: str = "",
    box_overlay: bool = True,
    point_size: float = 4.0,
    point_alpha: float = 0.85,
    jitter_width: float = 0.12,
    fig_width: float = 10.0,
    fig_height: Optional[float] = None,
    label_wrap: int = 14,
    label_rot: int = 35,
) -> plt.Figure:
    """
    Create replicate plot showing PSI values per replicate for each group.
    
    Parameters
    ----------
    replicate_data : Dict[Tuple[int, int], np.ndarray]
        Dictionary mapping (isoform_idx, group_idx) -> PSI values per replicate
    group_labels : List[str]
        Labels for groups (x-axis)
    isoform_labels : List[str]
        Labels for isoforms (row titles)
    title : str
        Overall plot title
    xlabel : str
        Label for x-axis
    box_overlay : bool
        If True, overlay boxplots on scatter points
    point_size : float
        Size of scatter points
    point_alpha : float
        Alpha transparency of points
    jitter_width : float
        Width of horizontal jitter for points
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)
    label_wrap : int
        Width for wrapping x-axis labels
    label_rot : int
        Rotation angle for x-axis labels
        
    Returns
    -------
    plt.Figure
        The matplotlib figure
    """
    # Determine number of isoforms
    n_isoforms = max((k[0] for k in replicate_data.keys()), default=-1) + 1
    if n_isoforms == 0:
        raise ValueError("No data in replicate_data")
    
    n_groups = len(group_labels)
    
    H = fig_height if fig_height is not None else max(3.5, 2.5 * n_isoforms)
    fig, axes = plt.subplots(
        n_isoforms, 1,
        figsize=(fig_width, H),
        squeeze=False
    )
    axes = axes.flatten()
    
    for i in range(n_isoforms):
        ax = axes[i]
        
        # Plot scatter points with jitter
        for j in range(n_groups):
            y = np.asarray(replicate_data.get((i, j), np.array([], float)), float)
            if y.size == 0:
                continue
            x = np.full_like(y, j, dtype=float) + (np.random.random(size=y.size) - 0.5) * jitter_width
            ax.plot(x, np.clip(y, 0.0, 1.0), "o", ms=point_size, alpha=point_alpha)
        
        # Overlay boxplot if requested
        if box_overlay:
            ys = []
            for j in range(n_groups):
                arr = np.asarray(replicate_data.get((i, j), np.array([], float)), float)
                ys.append(arr if arr.size else np.array([np.nan]))
            
            bp = ax.boxplot(
                ys, positions=np.arange(n_groups), widths=0.55,
                manage_ticks=False, patch_artist=True,
                medianprops=dict(color="#333333", linewidth=1.2),
                boxprops=dict(facecolor="none", edgecolor="#333333", linewidth=1.0),
                whiskerprops=dict(color="#333333", linewidth=1.0),
                capprops=dict(color="#333333", linewidth=1.0),
            )
            for b in bp["boxes"]:
                b.set_alpha(0.55)
        
        # Styling
        ax.set_xlim(-0.5, n_groups - 0.5)
        ax.set_ylim(0.0, 1.0)
        ax.set_ylabel("PSI", fontsize=9)
        ax.set_xticks(np.arange(n_groups))
        
        if i == n_isoforms - 1:
            ax.set_xticklabels(
                _wrap_labels(group_labels, width=label_wrap),
                rotation=label_rot,
                ha='right',
                fontsize=8
            )
            if xlabel:
                ax.set_xlabel(xlabel, fontsize=9)
        else:
            ax.set_xticklabels([])
        
        # Add isoform label on right side
        ax.text(1.02, 0.5, isoform_labels[i], 
                transform=ax.transAxes, 
                fontsize=9, va='center', rotation=-90)
    
    if title:
        fig.suptitle(title, fontsize=12, y=0.995)
    
    plt.tight_layout()
    return fig


def plot_isoform_replicates_from_adata(
    adata,
    group_col: str,
    replicate_col: str,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    epsilon: float = 1e-6,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    box_overlay: bool = True,
    point_size: float = 4.0,
    point_alpha: float = 0.85,
    jitter_width: float = 0.12,
    fig_width: float = 10.0,
    fig_height: Optional[float] = None,
    label_wrap: int = 14,
    label_rot: int = 35,
    show_version: bool = False,
) -> Tuple[plt.Figure, List[str], List[str], Dict[Tuple[int, int], np.ndarray]]:
    """
    Plot per-replicate PSI values for a gene's isoforms across groups.
    
    This function creates a multi-panel plot showing PSI variability across
    biological replicates for each group. Each row represents one isoform.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    replicate_col : str
        Column in adata.obs defining replicates (e.g., 'batch', 'sample_id')
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot.
        Mutually exclusive with gene_id + top_n.
    gene_id : str, optional
        Gene ID to plot. Used with top_n to select isoforms.
    top_n : int
        Number of top isoforms to plot (when using gene_id)
    epsilon : float
        Small value to avoid division by zero
    estimator : str
        PSI estimator ('pseudobulk', 'dirichlet', 'cell-mean', 'cell-median', 'coverage-weighted')
    dirichlet_alpha : float
        Dirichlet alpha parameter
    box_overlay : bool
        If True, overlay boxplots on scatter points
    point_size : float
        Size of scatter points
    point_alpha : float
        Alpha transparency of points
    jitter_width : float
        Width of horizontal jitter for points
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)
    label_wrap : int
        Width for wrapping x-axis labels
    label_rot : int
        Rotation angle for x-axis labels
    show_version : bool
        If True, show full transcript IDs with version suffix
        
    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs
    groups : List[str]
        List of group names
    replicate_data : Dict[Tuple[int, int], np.ndarray]
        Dictionary of replicate PSI values
        
    Examples
    --------
    # Plot replicate variability for Myl6 using gene_id
    fig, iso_ids, groups, data = plot_isoform_replicates_from_adata(
        adata,
        group_col="cell_type",
        replicate_col="batch",
        gene_id="Myl6",
        top_n=3
    )
    plt.show()
    
    # Plot specific transcripts
    fig, iso_ids, groups, data = plot_isoform_replicates_from_adata(
        adata,
        group_col="cell_type",
        replicate_col="batch",
        transcripts=["ENSMUST00000027000", "ENSMUST00000114041"]
    )
    plt.show()
    """
    # Compute replicate PSI data
    iso_ids, groups, replicate_data = _compute_replicate_psi_from_adata(
        adata,
        group_col=group_col,
        replicate_col=replicate_col,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        epsilon=epsilon,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
    )
    
    if not iso_ids:
        raise ValueError(f"No isoforms found for gene {gene_id}" if gene_id else "No isoforms found")
    
    # Format isoform labels
    def _strip_version(tid: str) -> str:
        """Remove version suffix from transcript ID (e.g., ENST00000123.4 -> ENST00000123)."""
        i = tid.rfind(".")
        return tid[:i] if i > 0 and tid[i+1:].isdigit() else tid
    
    isoform_labels = iso_ids if show_version else [_strip_version(tid) for tid in iso_ids]
    
    # Get title - use gene_id if provided, otherwise use first transcript
    title_gene = gene_id if gene_id else iso_ids[0]
    
    # Create plot
    fig = _standalone_replicate_plot(
        replicate_data,
        group_labels=groups,
        isoform_labels=isoform_labels,
        title=f"{title_gene} - Replicate PSI",
        xlabel=group_col,
        box_overlay=box_overlay,
        point_size=point_size,
        point_alpha=point_alpha,
        jitter_width=jitter_width,
        fig_width=fig_width,
        fig_height=fig_height,
        label_wrap=label_wrap,
        label_rot=label_rot,
    )
    
    return fig, iso_ids, groups, replicate_data


In [ ]:
#| export
def plot_isoform_violin(
    adata,
    group_col: str,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    gene_col: str = "geneId",
    layer: str | None = None,
    log1p: bool = True,
    drop_zeros: bool = False,
    min_cells_per_group: int = 10,
    figsize: tuple[float, float] | None = None,
    figsize_per_panel: tuple[float, float] = (4.0, 4.0),
    palette: str | list = "tab10",
    stripplot: bool = True,
    point_size: float = 2.0,
    point_alpha: float = 0.6,
    jitter: float = 0.25,
    sharey: bool = True,
    title: str | None = None,
    show_version: bool = False,
) -> Tuple[plt.Figure, List]:
    """
    Plot violin plots showing isoform expression distribution across groups.
    
    Creates multi-panel violin plots for a gene's isoforms, with optional
    scatter points overlay. Supports both raw counts and log-transformed data.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object with transcript counts
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot.
        Mutually exclusive with gene_id + top_n.
    gene_id : str, optional
        Gene ID to plot. Used with top_n to select isoforms.
    top_n : int, default 2
        Number of top isoforms to plot by mean expression (when using gene_id)
    gene_col : str, default "geneId"
        Column in adata.var mapping transcripts to genes
    layer : str, optional
        Layer in adata.layers to use. If None, uses adata.X
    log1p : bool, default True
        Apply log1p transformation to counts
    drop_zeros : bool, default False
        Remove zero values before plotting
    min_cells_per_group : int, default 10
        Minimum cells required per group to include
    figsize : tuple, optional
        Overall figure size (width, height). If None, computed from figsize_per_panel
    figsize_per_panel : tuple, default (4.0, 4.0)
        Size per isoform panel (width, height)
    palette : str or list, default "tab10"
        Color palette for groups
    stripplot : bool, default True
        Overlay scatter points on violins
    point_size : float, default 2.0
        Size of scatter points
    point_alpha : float, default 0.6
        Alpha transparency of points
    jitter : float, default 0.25
        Horizontal jitter width for points
    sharey : bool, default True
        Share y-axis across panels
    title : str, optional
        Custom plot title. If None, uses gene_id or first transcript
    show_version : bool, default False
        Show full transcript IDs with version suffix
        
    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    axes : List[plt.Axes]
        List of axes objects
        
    Examples
    --------
    # Plot top 3 isoforms for Myl6 using gene_id
    fig, axes = plot_isoform_violin(
        adata,
        group_col="cell_type",
        gene_id="Myl6",
        top_n=3,
        stripplot=True,
    )
    plt.show()
    
    # Plot specific transcripts
    fig, axes = plot_isoform_violin(
        adata,
        group_col="cell_type",
        transcripts=["ENSMUST00000027000", "ENSMUST00000114041"],
        log1p=False,
        stripplot=False,
    )
    plt.show()
    """
    if group_col not in adata.obs:
        raise ValueError(f"group_col='{group_col}' not found in adata.obs")
    
    if gene_col not in adata.var.columns:
        raise ValueError(f"adata.var['{gene_col}'] not found")
    
    # Resolve transcripts using standard pattern
    isoforms = _resolve_transcripts(
        adata,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        group_col=group_col,
    )
    
    if not isoforms:
        raise ValueError(f"No isoforms found for gene_id='{gene_id}'" if gene_id else "No isoforms found")
    
    # Verify all transcripts exist
    missing = [t for t in isoforms if t not in adata.var_names]
    if missing:
        raise ValueError(f"Transcripts not found in adata.var_names: {missing}")
    
    # Extract expression
    if layer is None:
        Xsub = adata[:, isoforms].X
    else:
        if layer not in adata.layers:
            raise ValueError(f"layer='{layer}' not found in adata.layers")
        Xsub = adata[:, isoforms].layers[layer]
    
    Xsub = Xsub.toarray() if hasattr(Xsub, "toarray") else np.asarray(Xsub)
    Xsub = np.asarray(Xsub, dtype=float)
    
    if log1p:
        Xsub = np.log1p(Xsub)
    
    # Format isoform labels
    def _strip_version(tid: str) -> str:
        """Remove version suffix from transcript ID."""
        i = tid.rfind(".")
        return tid[:i] if i > 0 and tid[i+1:].isdigit() else tid
    
    isoform_labels = isoforms if show_version else [_strip_version(tid) for tid in isoforms]
    
    # Create long dataframe
    groups = adata.obs[group_col].astype(str).values
    df = pd.DataFrame(Xsub, columns=isoform_labels)
    df[group_col] = groups
    df = df[df[group_col].str.strip() != ""]
    
    long = df.melt(id_vars=group_col, var_name="isoform", value_name="expression")
    
    if drop_zeros:
        long = long[long["expression"] > 0]
    
    # Filter by minimum cells per group
    vc = df[group_col].value_counts()
    keep_groups = vc[vc >= int(min_cells_per_group)].index
    long = long[long[group_col].isin(keep_groups)]
    
    # Remove isoforms with no expression
    long = long.groupby("isoform", observed=False).filter(lambda g: g["expression"].sum() > 0)
    if long.empty:
        raise ValueError("Nothing left to plot after filtering")
    
    long[group_col] = pd.Categorical(long[group_col], categories=list(keep_groups), ordered=True)
    long["isoform"] = pd.Categorical(long["isoform"], categories=isoform_labels, ordered=True)
    
    # Plot setup
    sns.set_theme(style="white", context="notebook")
    
    feats = [f for f in isoform_labels if f in long["isoform"].unique()]
    n = len(feats)
    
    if n == 0:
        raise ValueError("No isoforms remaining after filtering")
    
    # Determine figure size
    if figsize is None:
        figsize = (figsize_per_panel[0] * n, figsize_per_panel[1])
    
    fig, axes = plt.subplots(
        1, n,
        figsize=figsize,
        sharey=bool(sharey),
    )
    if n == 1:
        axes = [axes]
    
    for i, feat in enumerate(feats):
        ax = axes[i]
        sub = long[long["isoform"] == feat]
        
        sns.violinplot(
            data=sub,
            x=group_col,
            y="expression",
            ax=ax,
            cut=0,
            inner=None,
            density_norm="width",
            palette=palette,
            linewidth=0.8,
        )
        
        if stripplot:
            sns.stripplot(
                data=sub,
                x=group_col,
                y="expression",
                ax=ax,
                color="k",
                size=float(point_size),
                alpha=float(point_alpha),
                jitter=float(jitter),
            )
        
        ax.set_title(str(feat), fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel("log1p(counts)" if (log1p and i == 0) else ("counts" if i == 0 else ""))
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        sns.despine(ax=ax)
    
    # Get title - use gene_id if provided, otherwise use first transcript
    title_gene = gene_id if gene_id else isoforms[0]
    fig.suptitle(title or f"{title_gene} isoforms", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    
    return fig, axes

In [ ]:
#| export
def plot_isoform_heatmap_percell(
    adata,
    group_col: str,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    epsilon: float = 1e-6,
    cell_subset: Optional[List[str]] = None,
    max_cells: Optional[int] = None,
    cluster_within_groups: bool = True,
    cmap: str = "magma",
    colorbar_label: str = "PSI",
    fig_width: float = 20.0,
    fig_height: Optional[float] = None,
    show_colorbar: bool = True,
    show_cell_labels: bool = False,
    show_group_boundaries: bool = True,
    show_version: bool = False,
) -> Tuple[plt.Figure, List[str], List[str], np.ndarray]:
    """
    Plot a per-cell heatmap for isoform PSI values.

    This function creates a heatmap where each column represents an individual
    cell (not aggregated by group). Cells are sorted by group and optionally
    clustered within each group to reveal similar patterns.

    Parameters
    ----------
    adata : AnnData
        Annotated data object with transcript counts
    group_col : str
        Column in adata.obs for grouping/sorting cells (e.g., 'cell_type')
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot.
        Mutually exclusive with gene_id + top_n.
    gene_id : str, optional
        Gene ID to plot. Used with top_n to select isoforms.
    top_n : int, default 2
        Number of top isoforms to include by mean expression (when using gene_id)
    epsilon : float, default 1e-6
        Small value to avoid division by zero
    cell_subset : List[str], optional
        Specific cell IDs to include. If None, uses all cells.
    max_cells : int, optional
        Maximum number of cells to plot. If exceeded, samples randomly within each group.
    cluster_within_groups : bool, default True
        Whether to hierarchically cluster cells within each group
    cmap : str, default "magma"
        Colormap name
    colorbar_label : str, default "PSI"
        Label for colorbar
    fig_width : float, default 20.0
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)
    show_colorbar : bool, default True
        Whether to show colorbar
    show_cell_labels : bool, default False
        Whether to show individual cell labels on x-axis
    show_group_boundaries : bool, default True
        Whether to show vertical lines between groups
    show_version : bool, default False
        Show full transcript IDs with version suffix

    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs
    cell_ids : List[str]
        List of cell IDs (in plot order)
    V : np.ndarray
        PSI matrix (isoforms × cells)

    Examples
    --------
    # Basic per-cell heatmap with clustering using gene_id
    fig, iso_ids, cell_ids, V = plot_isoform_heatmap_percell(
        adata,
        group_col="cell_type",
        gene_id="Myl6",
        top_n=3,
        max_cells=200,
    )
    plt.show()

    # Using specific transcripts
    fig, iso_ids, cell_ids, V = plot_isoform_heatmap_percell(
        adata,
        group_col="cell_type",
        transcripts=["ENSMUST00000027000", "ENSMUST00000114041"],
        max_cells=300,
    )
    plt.show()
    """
    from scipy.cluster.hierarchy import linkage, leaves_list
    from scipy.spatial.distance import pdist

    if "geneId" not in adata.var:
        raise ValueError("adata.var must contain 'geneId' column")

    if group_col not in adata.obs:
        raise ValueError(f"group_col '{group_col}' not found in adata.obs")

    # Resolve transcripts using the standard pattern
    iso_ids = _resolve_transcripts(
        adata,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        group_col=group_col,
    )
    
    if not iso_ids:
        raise ValueError(f"No isoforms found for gene {gene_id}" if gene_id else "No isoforms found")

    # Subset to specific cells if requested
    if cell_subset is not None:
        adata = adata[cell_subset, :].copy()

    # Sample cells within each group if too many
    if max_cells is not None and adata.n_obs > max_cells:
        groups = adata.obs[group_col].astype(str).values
        unique_groups = list(dict.fromkeys(groups))
        cells_per_group = max(1, max_cells // len(unique_groups))

        np.random.seed(42)
        keep_idx = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            if len(g_idx) > cells_per_group:
                g_idx = np.random.choice(g_idx, size=cells_per_group, replace=False)
            keep_idx.extend(g_idx)
        adata = adata[keep_idx, :].copy()

    # Create mask for resolved transcripts
    iso_mask = np.isin(adata.var_names, iso_ids)
    if not iso_mask.any():
        raise ValueError("None of the resolved transcripts found in adata.var_names")

    # Extract expression matrix
    X = _dense_X(adata.X)
    G = X[:, iso_mask]  # cells × isoforms

    # Calculate per-cell PSI
    lib = G.sum(1) + float(epsilon)
    PSI = np.divide(
        G,
        lib[:, None],
        out=np.zeros_like(G, float),
        where=(lib[:, None] > 0),
    )

    # Reorder columns to match iso_ids order
    var_names_subset = adata.var_names[iso_mask].tolist()
    col_order = [var_names_subset.index(tid) for tid in iso_ids]
    PSI = PSI[:, col_order]

    # Sort cells by group, with optional clustering within groups
    groups = adata.obs[group_col].astype(str).values
    unique_groups = list(dict.fromkeys(groups))

    cell_order = []
    group_boundaries = [0]

    for g in unique_groups:
        g_idx = np.where(groups == g)[0]

        if cluster_within_groups and len(g_idx) > 2:
            # Hierarchical clustering within group
            psi_subset = PSI[g_idx, :]

            # Only cluster if we have variance
            if psi_subset.std() > 1e-6:
                try:
                    dist = pdist(psi_subset, metric='euclidean')
                    if len(dist) > 0 and not np.all(dist == 0):
                        Z = linkage(dist, method='average')
                        cluster_order = leaves_list(Z)
                        g_idx = g_idx[cluster_order]
                except:
                    pass  # Keep original order if clustering fails

        cell_order.extend(g_idx)
        group_boundaries.append(len(cell_order))

    cell_order = np.array(cell_order)

    # Reorder PSI matrix
    V = PSI[cell_order, :].T  # isoforms × cells
    cell_ids = adata.obs_names[cell_order].tolist()
    sorted_groups = groups[cell_order]

    # Format isoform labels
    def _strip_version(tid: str) -> str:
        """Remove version suffix from transcript ID."""
        i = tid.rfind(".")
        return tid[:i] if i > 0 and tid[i+1:].isdigit() else tid

    row_labels = iso_ids if show_version else [_strip_version(tid) for tid in iso_ids]

    # Create heatmap
    n_rows, n_cols = V.shape
    H = fig_height if fig_height is not None else max(3.5, 0.35 * n_rows + 1.8)

    fig, ax = plt.subplots(figsize=(fig_width, H))
    im = ax.imshow(
        V,
        aspect="auto",
        interpolation="nearest",
        origin="lower",
        cmap=mpl.cm.get_cmap(cmap),
        vmin=0,
        vmax=1,
        extent=[-0.5, n_cols - 0.5, -0.5, n_rows - 0.5],
    )

    # Y-axis: isoforms
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(row_labels, fontsize=9)
    ax.set_ylabel("Isoform", fontsize=10)

    # X-axis: cells (no individual labels)
    ax.set_xticks([])
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(-0.5, n_rows - 0.5)

    # Get title - use gene_id if provided, otherwise use first transcript
    title_gene = gene_id if gene_id else iso_ids[0]
    ax.set_title(f"{title_gene} - Per-Cell Heatmap", fontsize=12, fontweight="bold", pad=15)

    if show_colorbar:
        cb = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
        cb.set_label(colorbar_label, fontsize=9)

    fig.tight_layout()

    # Re-apply xlim after tight_layout to ensure alignment
    ax.set_xlim(-0.5, n_cols - 0.5)

    # Group boundaries and colored boxes - AFTER tight_layout
    if show_group_boundaries and len(group_boundaries) > 2:
        from matplotlib.patches import Rectangle

        # Create color palette for groups
        unique_grps = list(dict.fromkeys(sorted_groups))
        cmap_tab = mpl.cm.get_cmap("tab10")
        grp_colors = {g: cmap_tab(idx % 10) for idx, g in enumerate(unique_grps)}

        # Draw white boundary lines
        for boundary in group_boundaries[1:-1]:
            ax.axvline(boundary - 0.5, color="white", linewidth=1.5, alpha=0.8)

        # Draw colored boxes and labels for each group
        box_height = 0.06  # Much thinner bar

        # Draw thin colored bar directly above heatmap
        for i in range(len(group_boundaries) - 1):
            start = group_boundaries[i]
            end = group_boundaries[i + 1]
            group_name = sorted_groups[start]

            # Draw thin colored rectangle directly above heatmap
            color = grp_colors.get(group_name, "gray")
            rect = Rectangle(
                (start - 0.5, n_rows - 0.5),
                end - start,
                box_height,
                facecolor=color, edgecolor="none", alpha=0.7,
                clip_on=False,
                transform=ax.transData,
            )
            ax.add_patch(rect)

        # Add labels below heatmap with counts
        for i in range(len(group_boundaries) - 1):
            start = group_boundaries[i]
            end = group_boundaries[i + 1]
            mid = (start + end) / 2
            group_name = sorted_groups[start]
            n_cells_in_group = end - start

            group_label = f"{group_name}\n(n={n_cells_in_group})"
            ax.text(
                mid, -0.7, group_label,
                ha="center", va="top", fontsize=8,
                rotation=45, fontstyle="italic"
            )

    return fig, iso_ids, cell_ids, V

In [ ]:
#| export
def plot_isoform_stacked_bar(
    adata,
    group_col: str,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: Optional[int] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    cmap: str = "tab10",
    fig_width: float = 12.0,
    fig_height: float = 6.0,
    label_wrap: int = 14,
    label_rot: int = 45,
    show_legend: bool = True,
    show_values: bool = False,
    show_version: bool = False,
) -> Tuple[plt.Figure, List[str], List[str], np.ndarray]:
    """
    Plot stacked bar chart showing isoform composition per group.

    Creates a stacked bar chart where each bar represents a group (e.g., cell type)
    and shows the percent composition of each isoform within that group.

    Parameters
    ----------
    adata : AnnData
        Annotated data object with transcript counts
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot.
        Mutually exclusive with gene_id + top_n.
    gene_id : str, optional
        Gene ID to plot. Used with top_n to select isoforms.
    top_n : int, optional
        Number of top isoforms to show separately. Others grouped as "Other".
        If None when using gene_id, shows all isoforms.
    estimator : str, default "pseudobulk"
        PSI estimator ('pseudobulk', 'dirichlet', 'cell-mean', 'cell-median', 'coverage-weighted')
    dirichlet_alpha : float, default 0.5
        Dirichlet alpha parameter
    epsilon : float, default 1e-6
        Small value to avoid division by zero
    cmap : str, default "tab10"
        Colormap name for isoforms
    fig_width : float, default 12.0
        Figure width in inches
    fig_height : float, default 6.0
        Figure height in inches
    label_wrap : int, default 14
        Width for wrapping x-axis labels
    label_rot : int, default 45
        Rotation angle for x-axis labels
    show_legend : bool, default True
        Whether to show legend
    show_values : bool, default False
        Whether to show percentage values on bars
    show_version : bool, default False
        Show full transcript IDs with version suffix

    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs
    groups : List[str]
        List of group names
    V : np.ndarray
        PSI matrix (isoforms × groups), values sum to 1.0 per group

    Examples
    --------
    # Basic stacked bar chart using gene_id
    fig, iso_ids, groups, V = plot_isoform_stacked_bar(
        adata,
        group_col="cell_type",
        gene_id="Myl6",
        top_n=5,
    )
    plt.show()

    # Using specific transcripts
    fig, iso_ids, groups, V = plot_isoform_stacked_bar(
        adata,
        group_col="cell_type",
        transcripts=["ENSMUST00000027000", "ENSMUST00000114041"],
        show_values=True,
    )
    plt.show()
    """
    # Handle transcripts vs gene_id+top_n
    if transcripts is not None:
        # User provided explicit transcripts - use them directly
        iso_ids = transcripts if isinstance(transcripts, list) else [transcripts]
        
        # Verify all transcripts exist
        missing = [t for t in iso_ids if t not in adata.var_names]
        if missing:
            raise ValueError(f"Transcripts not found in adata.var_names: {missing}")
        
        # Compute PSI matrix for these specific transcripts
        iso_mask = np.isin(adata.var_names, iso_ids)
        X = _dense_X(adata.X)
        G = X[:, iso_mask]
        
        groups_all = adata.obs[group_col].astype(str).values
        groups = list(dict.fromkeys(groups_all))
        
        # Compute PSI per group
        V_rows = []
        for g in groups:
            m = (groups_all == g)
            if not m.any():
                V_rows.append(np.zeros(len(iso_ids), float))
                continue
            if estimator in ("pseudobulk", "dirichlet"):
                S = G[m].sum(0)
                if estimator == "dirichlet":
                    S = S + float(dirichlet_alpha)
                denom = float(S.sum()) + float(epsilon)
                V_rows.append((S / denom).ravel())
            else:
                lib = G[m].sum(1) + float(epsilon)
                psi = np.divide(G[m], lib[:, None], 
                               out=np.zeros_like(G[m], float),
                               where=(lib[:, None] > 0))
                if estimator == "cell-median":
                    V_rows.append(np.nanmedian(psi, axis=0))
                elif estimator == "coverage-weighted":
                    w = lib / (lib.sum() + 1e-12)
                    V_rows.append((psi * w[:, None]).sum(0))
                else:
                    V_rows.append(np.nanmean(psi, axis=0))
        
        V = np.vstack(V_rows).T  # isoforms × groups
        
        # Reorder to match iso_ids order
        var_names_subset = adata.var_names[iso_mask].tolist()
        col_order = [var_names_subset.index(tid) for tid in iso_ids]
        V = V[col_order, :]
        
    elif gene_id is not None:
        # Compute PSI matrix using gene_id (don't filter by top_n yet)
        iso_ids, groups, V = _compute_group_matrix_from_adata(
            adata,
            gene_id,
            group_col,
            top_n=None,  # Get all isoforms first
            estimator=estimator,
            dirichlet_alpha=dirichlet_alpha,
            epsilon=epsilon,
        )

        if not iso_ids or V.size == 0:
            return plt.figure(), [], [], np.zeros((0, 0), float)

        # Handle top_n filtering with "Other" category
        if top_n is not None and len(iso_ids) > int(top_n):
            # Sort by mean PSI across groups
            mean_psi = V.mean(axis=1)
            top_idx = np.argsort(mean_psi)[::-1][:int(top_n)]

            # Split into top isoforms and "other"
            top_iso_ids = [iso_ids[i] for i in top_idx]
            V_top = V[top_idx, :]

            # Sum remaining isoforms into "Other"
            other_idx = [i for i in range(len(iso_ids)) if i not in top_idx]
            if other_idx:
                V_other = V[other_idx, :].sum(axis=0, keepdims=True)
                iso_ids = top_iso_ids + ["Other"]
                V = np.vstack([V_top, V_other])
            else:
                iso_ids = top_iso_ids
                V = V_top
    else:
        raise ValueError("Must provide either 'transcripts' or 'gene_id'")

    # Format isoform labels
    def _strip_version(tid: str) -> str:
        """Remove version suffix from transcript ID."""
        if tid == "Other":
            return tid
        i = tid.rfind(".")
        return tid[:i] if i > 0 and tid[i+1:].isdigit() else tid

    isoform_labels = iso_ids if show_version else [_strip_version(tid) for tid in iso_ids]

    # Create figure
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # Get colormap
    n_iso = len(iso_ids)
    if isinstance(cmap, str):
        cmap_obj = mpl.cm.get_cmap(cmap)
        colors = [cmap_obj(i / max(1, n_iso - 1)) for i in range(n_iso)]
    else:
        colors = cmap

    # Create stacked bar chart
    x = np.arange(len(groups))
    bottoms = np.zeros(len(groups))

    bars = []
    for i, (iso_label, color) in enumerate(zip(isoform_labels, colors)):
        heights = V[i, :] * 100  # Convert to percentage
        bar = ax.bar(x, heights, bottom=bottoms, label=iso_label,
                     color=color, edgecolor='white', linewidth=0.5)
        bars.append(bar)

        # Add percentage values if requested
        if show_values:
            for j, (xpos, height, bottom) in enumerate(zip(x, heights, bottoms)):
                if height > 2:  # Only show if segment is large enough
                    y_pos = bottom + height / 2
                    ax.text(xpos, y_pos, f'{height:.1f}%',
                           ha='center', va='center', fontsize=7,
                           color='white', fontweight='bold')

        bottoms += heights

    # Styling
    ax.set_xticks(x)
    ax.set_xticklabels(
        _wrap_labels(groups, width=label_wrap),
        rotation=label_rot,
        ha='right',
        fontsize=9
    )
    ax.set_ylabel('Isoform Composition (%)', fontsize=10)
    ax.set_xlabel(group_col, fontsize=10)
    ax.set_ylim(0, 100)
    
    # Get title - use gene_id if provided, otherwise use first transcript
    title_gene = gene_id if gene_id else iso_ids[0]
    ax.set_title(f'{title_gene} - Isoform Composition by {group_col}',
                fontsize=12, fontweight='bold', pad=15)

    # Legend
    if show_legend:
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left',
                 frameon=True, fontsize=8)

    # Grid
    ax.yaxis.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fig.tight_layout()

    return fig, iso_ids, groups, V

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()